In [ ]:
# =============================================================================
# PART 4: THRESHOLD SIGNATURES & SMART CONTRACTS
# =============================================================================

class ThresholdSignature:

    def __init__(self, parties: List[str], threshold: int = 3):
        self.parties = parties
        self.threshold = threshold
        self.key_shares = {party: secrets.token_bytes(32) for party in parties}
        self.public_key = hashlib.sha256(b''.join(self.key_shares.values())).hexdigest()

    def partial_sign(self, party: str, message: str) -> Optional[str]:
        """Create partial signature"""
        if party not in self.parties:
            return None

        share = self.key_shares[party]
        partial = hashlib.sha256(f"{message}{share}".encode()).hexdigest()
        return f"{party}:{partial[:16]}"

    def combine_signatures(self, partial_sigs: List[str], message: str) -> Optional[str]:
        """
        Combine threshold signatures into final signature
        In reality: Lagrange interpolation on elliptic curve
        """
        if len(partial_sigs) < self.threshold:
            return None

        # Verify we have enough distinct parties
        parties_used = set()
        for sig in partial_sigs:
            party = sig.split(':')[0]
            parties_used.add(party)

        if len(parties_used) < self.threshold:
            return None

        # Simulate signature combination
        combined = hashlib.sha256(
            f"{message}{''.join(partial_sigs)}".encode()
        ).hexdigest()

        return f"Threshold({self.threshold}-of-{len(self.parties)}):{combined}"

class SmartContract:

    def __init__(self, blockchain: PermissionedBlockchain, crab: CRABStorage):
        self.chain = blockchain
        self.storage = crab
        self.threshold_sig = ThresholdSignature(
            ['PBoC', 'Supreme_Court', 'NDRC', 'SAMR', 'Bank_Consortium'],
            threshold=3
        )
        self.scores: Dict[str, int] = {}
        self.blacklist: Dict[str, Dict] = {}
        self.contract_code = self._get_contract_source()

    def _get_contract_source(self) -> str:
        """Simulate open-source contract code"""
        return '''
fn issue_record(hash: Hash, expiry: BlockNumber) -> Result {
    require(threshold_verify(3, msg.signers), "Insufficient authority");
    storage.create(hash, expiry);
    emit RecordIssued(hash, msg.signers, block.timestamp);
    Ok(())
}

fn update_score(account: AccountId, new_score: u16) -> Result {
    require(new_score <= 1000, "Score out of range");
    let old_score = storage.scores[account];
    storage.scores[account] = new_score;
    emit ScoreChanged {
        account: account,
        new_score: new_score,
        diff: new_score - old_score,
        commit_hash: GIT_COMMIT_HASH
    };
    Ok(())
}

fn logical_delete(hash: Hash) -> Result {
    require(block.number >= storage.expiry[hash], "Expiry not reached");
    require(threshold_verify(3, msg.signers), "Authority required");
    storage.burn(hash);
    emit RecordBurned(hash, block.number);
    Ok(())
}
'''

    def issue_blacklist(self, citizen_id: str, offense: Dict, signers: List[str]) -> Dict:
        """
        Issue blacklist record with threshold signatures
        Requires 3-of-5 authority signatures
        """
        print(f"\n  Smart Contract: issue_blacklist({citizen_id})")
        print(f"  Required signers: 3-of-5 from {self.threshold_sig.parties}")

        # Collect partial signatures
        message = f"blacklist:{citizen_id}:{json.dumps(offense, sort_keys=True)}"
        partial_sigs = []

        for signer in signers:
            if signer in self.threshold_sig.parties:
                psig = self.threshold_sig.partial_sign(signer, message)
                partial_sigs.append(psig)
                print(f"    ✓ Partial signature from {signer}")

        if len(partial_sigs) < self.threshold_sig.threshold:
            return {'success': False, 'error': 'Insufficient signatures'}

        # Combine threshold signature
        final_sig = self.threshold_sig.combine_signatures(partial_sigs, message)

        # Create blockchain transaction
        tx = self.chain.create_transaction(
            sender="Supreme_Court",
            operation="issue_blacklist",
            data={
                'citizen_id': citizen_id,
                'offense': offense,
                'threshold_sig': final_sig
            }
        )
        self.chain.pending_transactions.append(tx)

        # Store in CRAB
        record_hash = self.storage.create(citizen_id, offense)

        return {
            'success': True,
            'tx_id': tx.tx_id,
            'record_hash': record_hash,
            'threshold_sig': final_sig,
            'gas_used': 46000,  # ~46k gas
            'cost_yuan': 0.002
        }

    def update_score(self, citizen_id: str, new_score: int, commit_hash: str) -> Dict:
        """
        Update credit score with full audit trail
        """
        old_score = self.scores.get(citizen_id, 650)
        diff = new_score - old_score

        tx = self.chain.create_transaction(
            sender="PBoC_Score_Oracle",
            operation="update_score",
            data={
                'citizen_id': citizen_id,
                'old_score': old_score,
                'new_score': new_score,
                'diff': diff,
                'commit_hash': commit_hash
            }
        )
        self.chain.pending_transactions.append(tx)
        self.scores[citizen_id] = new_score

        return {
            'tx_id': tx.tx_id,
            'old_score': old_score,
            'new_score': new_score,
            'diff': diff,
            'commit_hash': commit_hash,
            'gas_used': 46000,
            'cost_yuan': 0.002,
            'replayable': True
        }

    def logical_delete(self, record_id: str, signers: List[str], current_block: int) -> Dict:
        """
        Execute logical deletion (burn) with authority verification
        """
        message = f"burn:{record_id}:{current_block}"
        partial_sigs = []

        for signer in signers:
            psig = self.threshold_sig.partial_sign(signer, message)
            if psig:
                partial_sigs.append(psig)

        if len(partial_sigs) < 3:
            return {'success': False, 'error': 'Insufficient authority for deletion'}

        success = self.storage.burn(record_id, current_block)

        return {
            'success': success,
            'signatures_used': len(partial_sigs),
            'block': current_block,
            'status': 'burned' if success else 'active'
        }

print("\n" + "=" * 70)
print("PART 4: SMART CONTRACTS & THRESHOLD SIGNATURES")
print("=" * 70)

contract = SmartContract(blockchain, crab)

# Issue blacklist with threshold signatures
print(f"\nScenario: Court issues blacklist for judgment defaulter")
offense = {
    "court": "Shenzhen Bao'an District Court",
    "case_number": "(2024)粤0306执1234号",
    "judgment_amount": 1000000,
    "unpaid_amount": 1000000,
    "default_date": "2024-01-15",
    "violation_type": "refuse_to_comply"
}

result = contract.issue_blacklist(
    "defaulter_440306_1990_002",
    offense,
    signers=['Supreme_Court', 'NDRC', 'PBoC']  # 3-of-5
)

print(f"\n  Transaction result:")
print(f"    Success: {result['success']}")
print(f"    TX ID: {result['tx_id']}")
print(f"    Gas used: {result['gas_used']:,}")
print(f"    Cost: ¥{result['cost_yuan']:.3f}")
print(f"    Threshold sig: {result['threshold_sig'][:40]}...")

# Update score
print(f"\nScenario: Score update after payment")
score_result = contract.update_score(
    "citizen_440306_1990_001",
    new_score=720,
    commit_hash="a1b2c3d4e5f6"  # Git commit hash
)
print(f"  Score changed: {score_result['old_score']} → {score_result['new_score']}")
print(f"  Diff: {score_result['diff']:+d}")
print(f"  Commit hash: {score_result['commit_hash']}")
print(f"  Replayable off-chain: {score_result['replayable']}")

# Mine block
print(f"\nMining block with {len(blockchain.pending_transactions)} transactions...")
block = blockchain.mine_block(validator="Supreme_Court")
print(f"  Block #{block.index} mined by {block.validator}")
print(f"  Merkle root: {block.merkle_root[:32]}...")
print(f"  Hash: {block.hash[:32]}...")
print(f"  Finality time: 3.0s")


PART 4: SMART CONTRACTS & THRESHOLD SIGNATURES

Scenario: Court issues blacklist for judgment defaulter

  Smart Contract: issue_blacklist(defaulter_440306_1990_002)
  Required signers: 3-of-5 from ['PBoC', 'Supreme_Court', 'NDRC', 'SAMR', 'Bank_Consortium']
    ✓ Partial signature from Supreme_Court
    ✓ Partial signature from NDRC
    ✓ Partial signature from PBoC

  Transaction result:
    Success: True
    TX ID: d3b87893c3754b65
    Gas used: 46,000
    Cost: ¥0.002
    Threshold sig: Threshold(3-of-5):70ea9d28ad4eaf57a6ef5c...

Scenario: Score update after payment
  Score changed: 650 → 720
  Diff: +70
  Commit hash: a1b2c3d4e5f6
  Replayable off-chain: True

Mining block with 2 transactions...
  Block #1 mined by Supreme_Court
  Merkle root: 3b0b0eb9e10eec917d091ab5a22104c7...
  Hash: 50ceb60889669de861cc3dce18cf9293...
  Finality time: 3.0s
